In [1]:
from typing import Any, Callable, Optional, Union

from pprint import pprint
from datetime import datetime
from pathlib import Path
import os
import re

from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import pytz
import numpy as np
import safetensors.torch as safetensors
import tqdm.notebook as tqdm
import torch
import torch.utils.data as torchdata
import torch.nn as nn
import torchmetrics
import segmentation_models_pytorch as torchseg
import yaml

from flatiron.core.dataset import Dataset
from flatiron.core.types import Compiled, Filepath, Getter
from flatiron.core.tools import get_tensorboard_project
from flatiron.torch.tools import ModelCheckpoint, get_callbacks, TorchDataset, _execute_epoch

Filepath = Union[str, Path]

# QUADRO P6000 is CUDA 6.1
# Triton is CUDA 7.0+
# This tells triton to shutup
import torch._dynamo
torch._dynamo.config.suppress_errors = True

2025-02-27 23:29:23.912914: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740698963.931440 1626534 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740698963.936890 1626534 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-27 23:29:23.954802: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def train(
    device,      # type: str
    model,       # type: torch.nn.Module
    optimizer,   # type: torch.optim.Optimizer
    loss,        # type: torch.nn.Module
    metrics,     # type: list[torch.nn.Module]
    callbacks,   # type: Callbacks
    train_data,  # type: Dataset
    test_data,   # type: Dataset
    params,      # type: dict
):
    # type: (...) -> None
    '''
    Train Torch model.

    Args:
        device (str): Device to compile to.
        model (torch.nn.Module): Model to be compiled.
        optimizer (dict): Optimizer config for compilation.
        loss (str): Loss to be compiled.
        metrics (list[str]): Metrics function to be compiled.
        callbacks (dict): Dict of callbacks.
        train_data (Dataset): Training dataset.
        test_data (Dataset): Test dataset.
        params (dict): Training params.
    '''
    checkpoint = callbacks['checkpoint']  # type: Any
    writer = callbacks['tensorboard']
    batch_size = params['batch_size']

    device = torch.device(device)
    torch.manual_seed(params['seed'])
    model = model.to(device)
    loss = loss.to(device)
    metrics = [x.to(device) for x in metrics]

    train_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(train_data), batch_size=batch_size
    )  # type: torchdata.DataLoader
    test_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(test_data), batch_size=batch_size
    )  # type: torchdata.DataLoader

    kwargs = dict(
        model=model,
        optimizer=optimizer,
        loss_func=loss,
        device=device,
        metrics_funcs=metrics,
        writer=writer,
    )
    for i in tqdm.trange(params['epochs']):
        _execute_epoch(
            epoch=i, mode='train', data_loader=train_loader,
            checkpoint=checkpoint, **kwargs
        )
        _execute_epoch(epoch=i, mode='test', data_loader=test_loader, **kwargs)
        if checkpoint.save_freq == 'epoch':
            checkpoint.save(model, i)

In [3]:
# DATA
data_kwargs = dict(
    directory='/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
    label_axis=-1,
    labels=['a'],
)
data = Dataset.read_directory(**data_kwargs)
train_data, test_data = data.train_test_split()
print('DATA')
pprint(data_kwargs)

DATA
{'directory': '/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
 'label_axis': -1,
 'labels': ['a']}


In [4]:
# MODEL
class Conv2DBlock(nn.Module):
    def __init__(self, in_channels, filters=16, dtype=torch.float16):
        super().__init__()
        kwargs = dict(
            out_channels=filters, kernel_size=(3, 3),
            stride=(1, 1), padding=1, padding_mode='reflect', dtype=dtype
        )
        self.conv_1 = nn.Conv2d(in_channels=in_channels, **kwargs)
        self.act_1 = nn.ReLU()
        self.batch_1 = nn.BatchNorm2d(filters, dtype=dtype)
        self.act_1 = nn.Sigmoid()
        self.conv_2 = nn.Conv2d(in_channels=filters, **kwargs)
        self.act_2 = nn.ReLU()
        self.batch_2 = nn.BatchNorm2d(filters, dtype=dtype)

    def forward(self, x):
        x = self.conv_1(x)
        x = self.act_1(x)
        x = self.batch_1(x)
        x = self.conv_2(x)
        x = self.act_2(x)
        x = self.batch_2(x)
        return x


class AtttentionGate2DBlock(nn.Module):
    def __init__(self, in_channels, filters=16, dtype=torch.float16):
        super().__init__()
        kwargs = dict(
            kernel_size=(3, 3),
            stride=(1, 1), padding=1, padding_mode='reflect', dtype=dtype
        )
        self.conv_0 = nn.Conv2d(in_channels=in_channels, out_channels=filters, **kwargs)
        self.conv_1 = nn.Conv2d(in_channels=in_channels, out_channels=filters, **kwargs)
        self.act_1 = nn.ReLU()
        self.conv_2 = nn.Conv2d(in_channels=filters, out_channels=1, **kwargs)
        self.act_1 = nn.Sigmoid()

    def forward(self, skip_connection, query):
        skip = self.conv_0(skip_connection)
        query = self.conv_1(query)

        gate = torch.add(skip, query)
        gate = self.act_1(gate)
        gate = self.conv_2(gate)
        gate = self.act_2(gate)
        gate = torch.multiply(skip, gate)

        x = torch.concatenate([gate, query])
        return x


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, attention=False, dtype=torch.float16):
        super().__init__()
        self._attention = attention
        kwargs = dict(dtype=dtype)
        io_kwargs = dict(kernel_size=(1, 1), stride=(1, 1), dtype=dtype)
        pool_kwargs = dict(kernel_size=(2, 2), stride=(2, 2))
        trans_kwargs = dict(kernel_size=(2, 2), stride=(2, 2), dtype=dtype)
        
        # self.input = nn.Conv2d(in_channels=in_channels, out_channels=16, **io_kwargs)

        self.encode_block_00 = Conv2DBlock(in_channels=in_channels, filters=16, **kwargs)
        self.downsample_00   = nn.MaxPool2d(**pool_kwargs)
        self.encode_block_01 = Conv2DBlock(in_channels=16, filters=32, **kwargs)
        self.downsample_01   = nn.MaxPool2d(**pool_kwargs)

        self.middle_block    = Conv2DBlock(in_channels=32, filters=64, **kwargs)

        self.upsample_00     = nn.ConvTranspose2d(in_channels=64, out_channels=32, **trans_kwargs)  # concat with encode_block_01
        self.decode_block_00 = Conv2DBlock(in_channels=64, filters=32, **kwargs)
        self.upsample_01     = nn.ConvTranspose2d(in_channels=32, out_channels=16, **trans_kwargs)  # concat with encode_block_00
        self.decode_block_01 = Conv2DBlock(in_channels=32, filters=out_channels, **kwargs)

        self.output = nn.Sigmoid()
        # self.output = nn.Conv2d(in_channels=16, out_channels=out_channels, **io_kwargs)
        
        # if attention:
        #     self.decode_atten_4 = AtttentionGate2DBlock(in_channels=256, filters=256, **kwargs)
        #     self.decode_atten_3 = AtttentionGate2DBlock(in_channels=256, filters=128, **kwargs)
        #     self.decode_atten_2 = AtttentionGate2DBlock(in_channels=128, filters=64, **kwargs)
        #     self.decode_atten_1 = AtttentionGate2DBlock(in_channels=64, filters=32, **kwargs)

    def forward(self, x):
        # x = self.input(x)
        x0 = self.encode_block_00(x)
        x = self.downsample_00(x0)
        x1 = self.encode_block_01(x)
        x = self.downsample_01(x1)
        x = self.middle_block(x)

        x = self.upsample_00(x)
        x = torch.concatenate([x, x1], axis=1)
        x = self.decode_block_00(x)
        x = self.upsample_01(x)
        x = torch.concatenate([x, x0], axis=1)
        x = self.decode_block_01(x)
        x = self.output(x)
        return x

    
model_kwargs = dict(
    in_channels=3,
    out_channels=1,
)
model = UNet(**model_kwargs)

# x = torch.rand((1, 3, 208, 208), dtype=torch.float16)
# model(x).shape
print('MODEL')
pprint(model_kwargs)

MODEL
{'in_channels': 3, 'out_channels': 1}


In [5]:
# # LOSS
class JaccardLoss(nn.Module):
    def __init__(self, smooth=0, dtype=torch.float16):
        super().__init__()
        self._smooth = smooth
        self._dtype = dtype

    def forward(self, y_pred, y):
        smooth = self._smooth
        dtype = self._dtype
        i = torch.sum(torch.abs(y * y_pred), dtype=dtype)
        u = torch.sum(torch.abs(y) + torch.abs(y_pred), dtype=dtype)
        jacc = (i + smooth) / (u - i + smooth)
        loss = (1 - jacc)
        return loss
    
# # class JaccardLoss(nn.modules.loss._Loss):
# #     def __init__(self, eps=1e-7, smooth=0):
# #         super().__init__()
# #         self._eps = eps
# #         self._smooth = smooth

# #     def forward(self, y_pred, y_true):
# #         eps = self._eps
# #         smooth = self._smooth

# #         i = (y_pred * y_true).sum(dim=(2, 3))
# #         u = (y_pred + y_true).sum(dim=(2, 3)) - i
# #         jacc = (i + smooth + eps) / (u + smooth + eps)
# #         loss = 1 - jacc.mean()
# #         return loss
    
# y = torch.ones((8, 3, 10, 10), dtype=torch.float16) * 0.1
# y_pred = torch.ones((8, 3, 10, 10), dtype=torch.float16)
# loss = JaccardLoss()(y_pred, y)
# loss.backward()

In [6]:
# METRIC
# from torchmetrics.metric import Metric
# class Dice(Metric):
#     def __init__(self, smooth=1.0, dtype=torch.float16):
#         super().__init__()
#         self._smooth = smooth
#         self._dtype = dtype

#     @torch.no_grad()
#     def update(self, preds, target):
#         self._y_pred = preds
#         self._y = target

#     @torch.no_grad()
#     def compute(self):
#         smooth = self._smooth
#         y = self._y
#         y_pred = self._y_pred
#         i = torch.sum(y * y_pred)
#         u = torch.sum(y) + torch.sum(y_pred)
#         dice = (2.0 * i + smooth) / (u + smooth)
#         return dice

# class BinaryDice(torchmetrics.classification.Dice):        
#     def update(self, y_pred, y):        
#         binarize = Binarizer()
#         super().update(binarize(y_pred), binarize(y))
        
# class BinaryDiceLoss(nn.loss.Dice):
#     def __init__(self):
#         super().__init__()

#     def forward(self, y_pred, y):
#         binarize = Binarizer()
#         binarize(y_pred), binarize(y)
#         return 1 - super().compute()

# y = torch.rand((1, 1, 208, 208), dtype=torch.float16)
# y_pred = torch.rand((1, 1, 208, 208), dtype=torch.float16)

# x = BinaryDice()
# x.update(y_pred, y)
# print(x.compute())

# loss = BinaryDiceLoss()
# loss.forward(y_pred, y)


# class JaccardLoss(torchseg.losses.JaccardLoss):
#     def forward(self, y_pred, y, *args, **kwargs):
#         binarize = Binarizer()
#         return super().forward(binarize(y_pred), binarize(y), *args, **kwargs)

In [7]:
class MinMaxScaler:
    def __init__(self, start=0, stop=1, eps=1e-7):
        self.start = start
        self.stop = stop
        self.eps = eps

    def __call__(self, x):        
        min_ =  torch.amin(x)
        max_ =  torch.amax(x)
        delta = max_ - min_
        if delta == 0:
            return x
        unit = (x - min_) / (delta + self.eps)
        output = unit + self.start * (self.stop - self.start)
        return output
    
class Binarizer:
    def __call__(self, x):
        minmax = MinMaxScaler()
        return torch.round(minmax(x))

class BinaryDice(torchmetrics.classification.Dice):        
    def update(self, y_pred, y):
        binarize = Binarizer()
        y_pred = binarize(y_pred).type(torch.int)
        y = binarize(y).type(torch.int)
        super().update(y_pred, y)

In [78]:
def get_layers(model):
    layers = {x: getattr(model, x) for x in dir(model)}
    layers = filter(lambda x: isinstance(x[1], nn.Module), layers.items())
    return dict(layers)

def freeze_layers(model, include='^[a-z]', exclude='_orig_mod', unfreeze=False):
    layers = get_layers(model)
    keep = filter(lambda x: re.search(include, x), layers.keys())
    keep = filter(lambda x: not re.search(exclude, x), keep)
    keep = set(keep)
    for key, layer in layers.items():
        if key in keep:
            if unfreeze:
                print(f'Unfroze: {key}')
                for param in layer.parameters():
                    param.requires_grad = True
            else:
                print(f'  Froze: {key}')
                for param in layer.parameters():
                    param.requires_grad = False
        else:
            print(f'Skipped: {key}')

src = '/mnt/storage/projects/unet001/tensorboard/d-2025-02-27_t-18-29-30/models/p-unet001_d-2025-02-27_t-18-29-30_e-009.pth'
model = torch.load(src, weights_only=False)
freeze_layers(model, exclude='upsample_01|decode_block_01|output')
len(list(filter(lambda x: x.requires_grad, model.parameters())))

Skipped: _orig_mod
  Froze: decode_block_00
Skipped: decode_block_01
  Froze: downsample_00
  Froze: downsample_01
  Froze: encode_block_00
  Froze: encode_block_01
  Froze: middle_block
Skipped: output
  Froze: upsample_00
Skipped: upsample_01


10

In [79]:
# CALLBACKS
tb = get_tensorboard_project(
    project='unet001',
    root='/mnt/storage/projects',
    extension='pth',
    timezone='America/Detroit',
)
print('TENSORBOARD')
pprint(tb)

callback_kwargs = dict(
    log_directory=tb['log_dir'],
    checkpoint_pattern=tb['checkpoint_pattern'],
    checkpoint_params=dict(save_freq='epoch'),
)
callbacks = get_callbacks(**callback_kwargs)
print()
print('CALLBACKS')
pprint(callback_kwargs)

# TRAIN KWARGS
train_kwargs = dict(
    device='cuda',
    model=torch.compile(model),
    optimizer=torch.optim.SGD(
        filter(lambda x: x.requires_grad, model.parameters()),
        lr=0.002,
    ),
    # loss=JaccardLoss(),
    # loss=BinaryDiceLoss(),
    # loss=JaccardLoss(smooth=0),
    loss=nn.MSELoss(),
    metrics=[
        BinaryDice(),
        # BinaryDice(),
        # torchmetrics.Dice(),
        # nn.MSELoss(),
        # JaccardIndex(task='binary'),
    ],
    callbacks=callbacks,
    train_data=train_data,
    test_data=test_data,
    params=dict(
        epochs=10,
        seed=42,
        batch_size=16,
    )
)
print()
print('TRAIN')
pprint(train_kwargs)

# TRAIN
print()
train(**train_kwargs)

TENSORBOARD
{'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-27_t-19-54-46/models/p-unet001_d-2025-02-27_t-19-54-46_e-{epoch:03d}.pth',
 'log_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-27_t-19-54-46',
 'model_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-27_t-19-54-46/models',
 'root_dir': '/mnt/storage/projects/unet001/tensorboard'}

CALLBACKS
{'checkpoint_params': {'save_freq': 'epoch'},
 'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-27_t-19-54-46/models/p-unet001_d-2025-02-27_t-19-54-46_e-{epoch:03d}.pth',
 'log_directory': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-27_t-19-54-46'}

TRAIN
{'callbacks': {'checkpoint': <flatiron.torch.tools.ModelCheckpoint object at 0x7fc04a449870>,
               'tensorboard': <torch.utils.tensorboard.writer.SummaryWriter object at 0x7fc04ca1a1d0>},
 'device': 'cuda',
 'loss': MSELoss(),
 'metrics': [BinaryDice()],
 'model': OptimizedModule(
  (_orig_mod):

  0%|          | 0/10 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
!exa --tree /mnt/storage/projects/unet001/tensorboard/d-2025*

/mnt/storage/projects/unet001/tensorboard/d-2025-02-25_t-13-44-28
├── events.out.tfevents.1740509068.5abe6464f7f1.258531.1
└── models
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-000.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-001.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-002.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-003.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-004.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-005.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-006.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-007.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-008.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-009.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-010.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-011.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-012.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-013.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28